# Qwen3.8 weight optimization: exploratory evidence

This notebook displays local receipts for issue #16. Set `QWEN38_WEIGHT_RUN_DIR` to the frozen run directory before starting the kernel, or use the default below. Receipts are snapshots: regenerate the evaluation report with the authenticated CLI before interpreting it. No candidate text or solutions are displayed.


In [ ]:
from pathlib import Path
import json, os

repo = Path.cwd().resolve()
while not (repo / 'research/rsi/qwen38-weight-execution-v1.json').is_file():
    if repo == repo.parent:
        raise FileNotFoundError('Run inside the issue 16 worktree')
    repo = repo.parent
run = Path(os.environ.get('QWEN38_WEIGHT_RUN_DIR', repo / 'artifacts/qwen38-weight-exploratory-20261109-v1'))
print('Run directory:', run)


## Collection and parameter update

The synthetic canary demonstrated that the local training path can change and reload adapter tensors; it is not an ARC accuracy result. Collection counts below can be incomplete and are not the final, length-filtered training set.


In [ ]:
receipts = list((run / 'receipts').glob('*.json'))
if receipts:
    receipt = json.loads(max(receipts, key=lambda p: p.stat().st_mtime_ns).read_text())
    print('Latest collection receipt counts:', receipt.get('counts', 'unavailable'))
else:
    print('Collection: not run or no receipt available')

training = run / 'training'
for name in ('training-result-v1.json', 'training-decision-v1.json', 'training-failed-v1.json'):
    path = training / name
    if path.is_file():
        data = json.loads(path.read_text())
        print('Training:', {key: data.get(key) for key in ('status', 'mode', 'counts', 'changed_lora_tensors', 'max_abs_logit_delta', 'adapter_sha256') if key in data})
        break
else:
    print('Training: not run')


## Matched Gen0 / Gen1 evaluation

Scores are offline comparisons on the existing 50-task exploratory set, not an official held-out ARC score. The report tracks completed and incomplete tasks; two recorded attempts are not assumed independent. A missing report is not a zero score.

The recovery run uses a separate registration after the initial attempt stopped before model loading because the adapter directory was not private. The original failure receipt remains in `adapter-v1`; recovery reports are in `adapter-v2`.


In [ ]:
reports = list((run / 'evaluation/adapter-v2').glob('report-*.json'))
if not reports:
    reports = list((run / 'evaluation/adapter-v1').glob('report-*.json'))
if reports:
    report = json.loads(max(reports, key=lambda p: p.stat().st_mtime_ns).read_text())
    print('Report status:', report.get('status'), 'schedule:', report.get('schedule_status'))
    print('Training source:', report.get('training_mode'), report.get('training_source_counts'))
    paired = report.get('paired', {})
    for arm in ('baseline', 'gen1', 'paired'):
        print(arm, paired.get(arm, 'unavailable'))
    print('Report identity:', report.get('sha256'))
else:
    print('Gen1 evaluation: not run or no report available')


The historical FP8 fallback, if selected, makes this cross-run distillation rather than pure 4-bit self-generated training. A changed adapter demonstrates a parameter update, not improved accuracy; one generation and a paired exploratory comparison do not establish recursive self-improvement. Consult the run's source replay and training receipts for provenance and failure details.
